In [1]:
#1: imports + Spark
import time
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

from mastodon_utils import get_spark, read_table, write_table, DATA_DIR

spark = get_spark("mastodon-batch")

Spark 3.5.3 prêt - Spark UI : http://localhost:4040


In [2]:
#2:charger l’historique depuis PostgreSQL
toots = (
    read_table(spark, "toots")
    .dropDuplicates(["toot_id"]) 
    .withColumn("day", F.to_date("event_time"))
    .withColumn("hour", F.hour("event_time"))
    .withColumn("hashtag_list", F.split("hashtags", ","))
)
toots.cache()
print("toots :", toots.count(), "| utilisateurs :", toots.select("user_id").distinct().count())
toots.select("day").groupBy().agg(F.min("day").alias("premier jour"), F.max("day").alias("dernier jour")).show()

toots : 1049 | utilisateurs : 317
+------------+------------+
|premier jour|dernier jour|
+------------+------------+
|  2017-04-19|  2026-10-09|
+------------+------------+



In [3]:
#3:Transformation 1 utilisateurs actifs
MIN_TOOTS = 3

user_stats = (
    toots.groupBy("user_id", "username")
    .agg(
        F.count("*").alias("nb_toots"),
        F.round(F.avg("text_length"), 1).alias("avg_length"),
        F.sum("favourites_count").alias("total_favourites"),
        F.sum("reblogs_count").alias("total_reblogs"),
    )
)
active_users = user_stats.filter(F.col("nb_toots") >= MIN_TOOTS).orderBy(F.desc("nb_toots"))
active_toots = toots.join(active_users.select("user_id"), on="user_id", how="left_semi")

print(f"{active_users.count()} utilisateurs ont au moins {MIN_TOOTS} toots -> {active_toots.count()} toots")
active_users.show(10, truncate=False)

52 utilisateurs ont au moins 3 toots -> 752 toots
+------------------+---------------------+--------+----------+----------------+-------------+
|user_id           |username             |nb_toots|avg_length|total_favourites|total_reblogs|
+------------------+---------------------+--------+----------+----------------+-------------+
|115061140460610278|hackaday             |201     |1976.1    |18              |355          |
|116070447239053927|feed                 |113     |389.6     |3               |84           |
|116988275448638305|sipirtu              |82      |221.6     |2               |1            |
|116450756112437762|techwire             |41      |422.7     |5               |65           |
|113136422441996049|researchbuzz_firehose|25      |502.1     |3               |9            |
|116862964066274332|teeeeeeeee           |22      |224.7     |6               |23           |
|114434182937791007|hugovalters          |19      |242.7     |2               |4            |
|109683831

In [4]:
#4:Transformation 2 hashtags par jour + hashtag le plus fréquent
hashtags = (
    toots.select("toot_id", "day", F.explode("hashtag_list").alias("hashtag"))
    .filter(F.col("hashtag") != "")
)

top_hashtags = hashtags.groupBy("hashtag").agg(F.count("*").alias("nb_toots")).orderBy(F.desc("nb_toots"))
top = top_hashtags.first()
print(f"Hashtag le plus fréquent : #{top['hashtag']} ({top['nb_toots']} toots)")
top_hashtags.show(10)

hashtags_per_day = hashtags.groupBy("day", "hashtag").agg(F.count("*").alias("nb_toots"))

w = Window.partitionBy("day").orderBy(F.desc("nb_toots"))
top_hashtag_per_day = (
    hashtags_per_day.withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") == 1)
    .drop("rank")
    .orderBy("day")
)
top_hashtag_per_day.show(20)

Hashtag le plus fréquent : #ai (465 toots)
+---------------+--------+
|        hashtag|nb_toots|
+---------------+--------+
|             ai|     465|
|         python|     255|
|machinelearning|     253|
|       software|     244|
|    development|     203|
|    engineering|     202|
|    datascience|     202|
|      community|     202|
|         coding|     201|
|      inclusive|     201|
+---------------+--------+
only showing top 10 rows

+----------+--------------+--------+
|       day|       hashtag|nb_toots|
+----------+--------------+--------+
|2017-04-19|    breizhcamp|       1|
|2019-08-08|         scala|       1|
|2021-09-26|categorytheory|       1|
|2022-04-07|    janusgraph|       1|
|2022-08-26|   apachespark|       1|
|2022-09-26|   apachespark|       1|
|2022-11-05|    monitoring|       1|
|2022-11-06| apacheairflow|       1|
|2022-11-10|       jupyter|       1|
|2022-11-16|   apachekafka|       1|
|2022-12-17|    apachebeam|       1|
|2023-01-07|         scala|       1

In [5]:
#5: agregations toots par jour longueur moyenne engagement
daily = (
    toots.groupBy("day")
    .agg(
        F.count("*").alias("nb_toots"),
        F.countDistinct("user_id").alias("nb_users"),
        F.round(F.avg("text_length"), 1).alias("avg_length"),
        F.round(F.avg(F.col("favourites_count") + F.col("reblogs_count")), 2).alias("avg_engagement"),
    )
    .orderBy("day")
)
daily.show(30)

TOP_N_USERS = 15
top_user_ids = active_users.limit(TOP_N_USERS).select("user_id", "username")
user_hour_activity = (
    toots.join(top_user_ids, on=["user_id", "username"])
    .groupBy("username", "hour")
    .agg(F.count("*").alias("nb_toots"))
)

+----------+--------+--------+----------+--------------+
|       day|nb_toots|nb_users|avg_length|avg_engagement|
+----------+--------+--------+----------+--------------+
|2017-04-19|       1|       1|     149.0|           0.0|
|2019-08-08|       1|       1|     279.0|           0.0|
|2021-09-26|       1|       1|     208.0|           0.0|
|2022-04-07|       1|       1|     498.0|           1.0|
|2022-08-26|       1|       1|     504.0|           0.0|
|2022-09-26|       1|       1|     504.0|           0.0|
|2022-11-05|       1|       1|     186.0|           0.0|
|2022-11-06|       1|       1|     475.0|           2.0|
|2022-11-10|       1|       1|     284.0|           1.0|
|2022-11-16|       1|       1|     299.0|           2.0|
|2022-12-17|       1|       1|     530.0|           8.0|
|2023-01-07|       1|       1|     254.0|           0.0|
|2023-02-07|       1|       1|      92.0|           0.0|
|2023-03-03|       1|       1|     225.0|           0.0|
|2023-03-15|       1|       1| 

In [6]:
#6:sauvegarder les resultats dans PostgreSQL

write_table(daily.coalesce(1), "batch_daily", mode="overwrite")
write_table(top_hashtags.coalesce(1), "batch_top_hashtags", mode="overwrite")
write_table(top_hashtag_per_day.coalesce(1), "batch_top_hashtag_per_day", mode="overwrite")
write_table(hashtags_per_day.coalesce(1), "batch_hashtags_per_day", mode="overwrite")
write_table(active_users.coalesce(1), "batch_active_users", mode="overwrite")
write_table(user_hour_activity.coalesce(1), "batch_user_hour_activity", mode="overwrite")
print("tables batch_* écrites dans PostgreSQL")

tables batch_* écrites dans PostgreSQL


In [7]:
#7:optimisation preparation

s140_schema = StructType([
    StructField("target", IntegerType()), StructField("id", StringType()),
    StructField("date", StringType()), StructField("flag", StringType()),
    StructField("user", StringType()), StructField("text", StringType()),
])
CSV_PATH = f"{DATA_DIR}/training.1600000.processed.noemoticon.csv"

def load_tweets():
    return (spark.read.csv(CSV_PATH, schema=s140_schema, encoding="ISO-8859-1")
            .withColumn("text_length", F.length("text")))

benchmark = [] 

def timed(experiment, label, action):
    spark.sparkContext.setJobDescription(f"{experiment} - {label}")
    t0 = time.time()
    result = action()
    seconds = round(time.time() - t0, 2)
    benchmark.append((experiment, label, seconds))
    print(f"{experiment:22} | {label:35} | {seconds:6.2f} s")
    return result

def run_queries(df):
    df.groupBy("user").count().orderBy(F.desc("count")).limit(10).collect() # users les plus actifs
    df.groupBy("target").agg(F.avg("text_length")).collect()# longueur moyenne par label
    df.filter(F.lower("text").contains("love")).count()  # tweets avec "love"

In [8]:
#8 : Expérience1  cache()
tweets = load_tweets()
timed("1. cache", "sans cache (3 requêtes)", lambda: run_queries(tweets))

tweets_cached = load_tweets().cache()
timed("1. cache", "mise en cache (1er count)", lambda: tweets_cached.count())
timed("1. cache", "avec cache (3 requêtes)", lambda: run_queries(tweets_cached))

1. cache               | sans cache (3 requêtes)             |   8.70 s
1. cache               | mise en cache (1er count)           |   6.61 s
1. cache               | avec cache (3 requêtes)             |   1.65 s


In [9]:
#9:Expérience 2 repartition()
tweets_cached.unpersist()
cores = spark.sparkContext.defaultParallelism
print("coeurs disponibles :", cores)

#cas 1:toutes les données dans 1 seule partition -> un seul coeur travaille
tweets_one = load_tweets().coalesce(1).cache()
tweets_one.count()
timed("2. repartition", f"{tweets_one.rdd.getNumPartitions()} partition", lambda: run_queries(tweets_one))

#cas 2: on recoupe en plus de partitions que de coeurs -> tous les coeurs travaillent
tweets_repart = tweets_one.repartition(cores).cache()
tweets_repart.count()
tweets_one.unpersist()
timed("2. repartition", f"{tweets_repart.rdd.getNumPartitions()} partitions (repartition)", lambda: run_queries(tweets_repart))

coeurs disponibles : 4
2. repartition         | 1 partition                         |   2.26 s
2. repartition         | 4 partitions (repartition)          |   1.53 s


In [10]:
#10: Expérience 3 spark.sql.shuffle.partitions
for n in [200, 8]:
    spark.conf.set("spark.sql.shuffle.partitions", n)
    timed("3. shuffle partitions", f"shuffle.partitions = {n}", lambda: run_queries(tweets_repart))
spark.conf.set("spark.sql.shuffle.partitions", 8)

3. shuffle partitions  | shuffle.partitions = 200            |   1.22 s
3. shuffle partitions  | shuffle.partitions = 8              |   1.03 s


In [11]:
#11: Expérience 4 coalesce() avant d ecrire
result = tweets_repart.groupBy("user").agg(F.count("*").alias("nb_tweets"))
print("partitions du résultat :", result.rdd.getNumPartitions())

timed("4. coalesce", f"écriture {result.rdd.getNumPartitions()} partitions",
      lambda: result.write.mode("overwrite").parquet("/tmp/bench_no_coalesce"))
timed("4. coalesce", "écriture après coalesce(1)",
      lambda: result.coalesce(1).write.mode("overwrite").parquet("/tmp/bench_coalesce"))

import os
print("fichiers sans coalesce :", len([f for f in os.listdir("/tmp/bench_no_coalesce") if f.endswith(".parquet")]))
print("fichiers avec coalesce(1) :", len([f for f in os.listdir("/tmp/bench_coalesce") if f.endswith(".parquet")]))

partitions du résultat : 4
4. coalesce            | écriture 4 partitions               |   2.71 s
4. coalesce            | écriture après coalesce(1)          |   1.22 s
fichiers sans coalesce : 4
fichiers avec coalesce(1) : 1


In [12]:
#12: recap et sauvegarde des mesures
spark.sparkContext.setJobDescription(None)
bench_df = spark.createDataFrame(benchmark, ["experiment", "label", "seconds"])
bench_df.show(truncate=False)
write_table(bench_df.coalesce(1), "batch_benchmark", mode="overwrite")

tweets_repart.unpersist()

+---------------------+--------------------------+-------+
|experiment           |label                     |seconds|
+---------------------+--------------------------+-------+
|1. cache             |sans cache (3 requêtes)   |8.7    |
|1. cache             |mise en cache (1er count) |6.61   |
|1. cache             |avec cache (3 requêtes)   |1.65   |
|2. repartition       |1 partition               |2.26   |
|2. repartition       |4 partitions (repartition)|1.53   |
|3. shuffle partitions|shuffle.partitions = 200  |1.22   |
|3. shuffle partitions|shuffle.partitions = 8    |1.03   |
|4. coalesce          |écriture 4 partitions     |2.71   |
|4. coalesce          |écriture après coalesce(1)|1.22   |
+---------------------+--------------------------+-------+



DataFrame[target: int, id: string, date: string, flag: string, user: string, text: string, text_length: int]